In [1]:
import pyspark.sql.functions as F

from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType
from src.spark_session import get_spark
from src.utils.display_mimic import display
from src.utils.logger import get_logger

spark = get_spark()
logger = get_logger(__name__)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-41908d42-0348-46a3-a9d9-9f7bfee25c77;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.0.0/delta-spark_2.12-3.0.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.0.0!delta-spark_2.12.jar (583ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.0.0/delta-storage-3.0.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.0.0!delta-storage.jar (54ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (83ms)
:: resolution report :: resolve 3219ms :: artifacts dl 73

In [4]:
df_products = spark.table('raw_product')

In [9]:
def build_store_product(df: DataFrame) -> DataFrame:

    df_clean = (
        df.groupBy(
            "ProductID",
            "ProductDesc",
            "ProductNumber",
            "MakeFlag",
            "Color",
            "SafetyStockLevel",
            "ReorderPoint",
            "Size",
            "SizeUnitMeasureCode",
            "WeightUnitMeasureCode"
        )
        .agg(
            F.first("StandardCost", ignorenulls=True).alias("StandardCost"),
            F.first("ListPrice", ignorenulls=True).alias("ListPrice"),
            F.first("Weight", ignorenulls=True).alias("Weight"),
            F.first("ProductCategoryName", ignorenulls=True).alias("ProductCategoryName"),
            F.first("ProductSubCategoryName", ignorenulls=True).alias("ProductSubCategoryName")
        )
    )

    df_casted = (
        df_clean
        .withColumn("StandardCost", F.col("StandardCost").cast(DecimalType(18,2)))
        .withColumn("ListPrice", F.col("ListPrice").cast(DecimalType(18,2)))
        .withColumn("Weight", F.col("Weight").cast(DecimalType(18,2)))
    )

    return df_casted

In [26]:
df_products_cleaned = df_products.transform(build_store_product)

In [28]:
display(df_products_cleaned, 25)

,ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,StandardCost,ListPrice,Weight,ProductCategoryName,ProductSubCategoryName
0,680,"HL Road Frame - Black, 58",FR-R92B-58,True,Black,500,375,58,CM,LB,1059.31,1431.50,2.24,None,Road Frames
1,706,"HL Road Frame - Red, 58",FR-R92R-58,True,Red,500,375,58,CM,LB,1059.31,1431.50,2.24,None,Road Frames
2,707,"Sport-100 Helmet, Red",HL-U509-R,False,Red,4,3,None,None,None,13.09,34.99,None,None,Helmets
3,708,"Sport-100 Helmet, Black",HL-U509,False,Black,4,3,None,None,None,13.09,34.99,None,None,Helmets
4,709,"Mountain Bike Socks, M",SO-B909-M,False,White,4,3,M,None,None,3.40,9.50,None,None,Socks
5,710,"Mountain Bike Socks, L",SO-B909-L,False,White,4,3,L,None,None,3.40,9.50,None,None,Socks
6,711,"Sport-100 Helmet, Blue",HL-U509-B,False,Blue,4,3,None,None,None,13.09,34.99,None,None,Helmets
7,712,AWC Logo Cap,CA-1098,False,Multi,4,3,None,None,None,6.92,8.99,None,None,Caps
8,713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,False,Multi,4,3,S,None,None,38.49,49.99,None,Clothing,Jerseys
9,714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,False,Multi,4,3,M,None,None,38.49,49.99,None,Clothing,Jerseys


In [30]:
try:
    logger.info('Starting saving products cleaned')
    (
        df_products_cleaned.write
        .format('delta')
        .mode('overwrite')
        .option('path', '/app/data/lake/store/store_product')
        .saveAsTable('store_products')
    )
    logger.info('Products cleaned saved with success!!')
    
except Exception as e:
    logger.exception("Error during ingestion process.")
    raise e

2026-03-05 02:15:37 | INFO | __main__ | Starting saving products cleaned


26/03/05 02:15:51 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`default`.`store_products` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/03/05 02:15:51 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/03/05 02:15:52 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/03/05 02:15:52 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/05 02:15:52 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


2026-03-05 02:15:53 | INFO | __main__ | Products cleaned saved with success!!


In [33]:
spark.sql('SELECT * FROM store_products LIMIT 10').show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------
 ProductID              | 680                        
 ProductDesc            | HL Road Frame - Black, 58  
 ProductNumber          | FR-R92B-58                 
 MakeFlag               | true                       
 Color                  | Black                      
 SafetyStockLevel       | 500                        
 ReorderPoint           | 375                        
 Size                   | 58                         
 SizeUnitMeasureCode    | CM                         
 WeightUnitMeasureCode  | LB                         
 StandardCost           | 1059.31                    
 ListPrice              | 1431.50                    
 Weight                 | 2.24                       
 ProductCategoryName    | NULL                       
 ProductSubCategoryName | Road Frames                
-RECORD 1--------------------------------------------
 ProductID              | 706                        
 ProductDesc            | HL